# Preprocess the AMCAT graduate salary data
Load, clean, split, and scale the data for training.

In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
data = pd.read_csv("Engineering_graduate_salary.csv")
print("Shape:", data.shape)
print(data.head())
print("Null counts:")
print(data.isnull().sum())

Shape: (2998, 34)
       ID Gender         DOB  10percentage                       10board  \
0  604399      f  1990-10-22         87.80                          cbse   
1  988334      m  1990-05-15         57.00                          cbse   
2  301647      m  1989-08-21         77.33  maharashtra state board,pune   
3  582313      m  1991-05-04         84.30                          cbse   
4  339001      f  1990-10-30         82.00                          cbse   

   12graduation  12percentage                    12board  CollegeID  \
0          2009         84.00                       cbse       6920   
1          2010         64.50                       cbse       6624   
2          2007         85.17  amravati divisional board       9084   
3          2009         86.00                       cbse       8195   
4          2008         75.00                       cbse       4889   

   CollegeTier  ... MechanicalEngg ElectricalEngg  TelecomEngg  CivilEngg  \
0            1  ...  

In [4]:
drop_columns = ["ID", "DOJ", "DOL", "Designation", "JobCity", "DOB",
                "CollegeID", "CollegeCityID"]
data = data.drop(columns=drop_columns, errors="ignore")
text_columns = data.select_dtypes(include="object").columns
for column in text_columns:
    data[column] = data[column].str.strip()
numeric_columns = data.select_dtypes(include=np.number).columns
data[numeric_columns] = data[numeric_columns].replace(-1, np.nan)
for column in numeric_columns:
    data[column] = data[column].fillna(data[column].median())
for column in text_columns:
    data[column] = data[column].fillna(data[column].mode()[0])

In [5]:
y = data["Salary"]
X = data.drop(columns=["Salary"])
X = pd.get_dummies(X, drop_first=True)
feature_columns = list(X.columns)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_test_raw = X_test.copy()
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, "scaler.pkl")
joblib.dump(feature_columns, "feature_columns.pkl")
data_splits = {
    "X_train_scaled": X_train_scaled,
    "X_test_scaled": X_test_scaled,
    "y_train": y_train,
    "y_test": y_test,
    "X_test_raw": X_test_raw
}
joblib.dump(data_splits, "data_splits.pkl")
print("Preprocessing artifacts saved.")

Preprocessing artifacts saved.
